In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, pipeline, logging
from trl import SFTTrainer
from peft import LoraConfig, PeftModel, TaskType
import torch
import json
from datasets import load_dataset

In [4]:
import pandas as pd

df = pd.read_csv("train2.csv")
print(df.head())

   id  qid1  qid2                                          question1  \
0   0     1     2  What is the step by step guide to invest in sh...   
1   1     3     4  What is the story of Kohinoor (Koh-i-Noor) Dia...   
2   2     5     6  How can I increase the speed of my internet co...   
3   3     7     8  Why am I mentally very lonely? How can I solve...   
4   4     9    10  Which one dissolve in water quikly sugar, salt...   

                                           question2  is_duplicate  
0  What is the step by step guide to invest in sh...             0  
1  What would happen if the Indian government sto...             0  
2  How can Internet speed be increased by hacking...             0  
3  Find the remainder when [math]23^{24}[/math] i...             0  
4            Which fish would survive in salt water?             0  


In [5]:
df = pd.read_csv("train2.csv")

# keep only needed columns
df = df[["question1", "question2", "is_duplicate"]]

# convert to JSONL
df.to_json("train2.jsonl", orient="records", lines=True)

In [6]:
import json
from datasets import Dataset

# Load JSONL
with open("train2.jsonl", "r", encoding="utf-8") as f:
    data = [json.loads(line) for line in f if line.strip()]

# Convert into instruction format
for d in data:
    d["text"] = f"""### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: {d['question1']}
Question 2: {d['question2']}

### Response:
{"Yes" if d["is_duplicate"] == 1 else "No"}"""

# Create HF dataset
dataset = Dataset.from_list(data)

# Keep only text column (important)
dataset = dataset.remove_columns(
    ["question1", "question2", "is_duplicate"]
)

print(dataset)

Dataset({
    features: ['text'],
    num_rows: 50000
})


In [7]:
dataset['text']

Column(['### Instruction:\nDetermine if the two questions are semantically duplicate.\n\n### Input:\nQuestion 1: What is the step by step guide to invest in share market in india?\nQuestion 2: What is the step by step guide to invest in share market?\n\n### Response:\nNo', '### Instruction:\nDetermine if the two questions are semantically duplicate.\n\n### Input:\nQuestion 1: What is the story of Kohinoor (Koh-i-Noor) Diamond?\nQuestion 2: What would happen if the Indian government stole the Kohinoor (Koh-i-Noor) diamond back?\n\n### Response:\nNo', '### Instruction:\nDetermine if the two questions are semantically duplicate.\n\n### Input:\nQuestion 1: How can I increase the speed of my internet connection while using a VPN?\nQuestion 2: How can Internet speed be increased by hacking through DNS?\n\n### Response:\nNo', '### Instruction:\nDetermine if the two questions are semantically duplicate.\n\n### Input:\nQuestion 1: Why am I mentally very lonely? How can I solve it?\nQuestion 2: 

### Load model and tokenizer

In [8]:
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

# tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# base_model
base_model = AutoModelForCausalLM.from_pretrained(
    model_name, device_map="auto", torch_dtype=torch.float16)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [9]:
base_model

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

### Baseline Generation

In [10]:
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model=base_model,
    tokenizer=tokenizer
)

q1 = "What is AI?"
q2 = "What is artificial intelligence?"

prompt = f"""
### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: {q1}
Question 2: {q2}

### Response: {"Duplicate" if d["is_duplicate"] == 1 else "Not duplicate"}
"""

result = pipe(
    prompt,
    max_new_tokens=5,
    do_sample=False
)

print(result[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


[transformers] Both `max_new_tokens` (=5) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer LlamaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: What is AI?
Question 2: What is artificial intelligence?

### Response: Not duplicate

### Explan


### Configure LORA

In [11]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],  # Adjust based on model architecture
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

### Training Configuration

In [12]:
import os
os.environ["WANDB_DISABLED"] = "True"

In [13]:
training_args = TrainingArguments(
    output_dir="./tinyllama-duplicate-detector",

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,

    num_train_epochs=1,

    learning_rate=2e-4,

    logging_steps=20,

    save_strategy="epoch",
    report_to="none",

    fp16=True,

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    save_total_limit=2
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [14]:
def tokenize(example):
    return tokenizer(example["text"], truncation=True, max_length=64)


dataset = dataset.map(tokenize, batched=True)

Map:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [15]:
# Trainer
trainer = SFTTrainer(
    model=base_model,
    train_dataset=dataset,
    peft_config=lora_config,
    args=training_args,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [16]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
20,2.412860
40,2.376181
60,2.057853
80,1.468858
100,1.085011
120,0.947975
140,0.974304
160,0.931938
180,0.952114
200,1.017460


TrainOutput(global_step=6250, training_loss=0.910869049987793, metrics={'train_runtime': 7649.6918, 'train_samples_per_second': 6.536, 'train_steps_per_second': 0.817, 'total_flos': 1.959663876459725e+16, 'train_loss': 0.910869049987793})

In [18]:
trainer.model.save_pretrained("D:\Desktop\Quora_Duplicate_Project")
tokenizer.save_pretrained("D:\Desktop\Quora_Duplicate_Project")

('D:\\Desktop\\Quora_Duplicate_Project\\tokenizer_config.json',
 'D:\\Desktop\\Quora_Duplicate_Project\\chat_template.jinja',
 'D:\\Desktop\\Quora_Duplicate_Project\\tokenizer.json')

In [9]:
# Robust test: use the same prompt format you used during training
import re

q1 = "How can I lose weight naturally?"
q2 = "What are natural methods for weight loss?"

prompt = f"""### Instruction:
Determine if the two questions are semantically duplicate.

### Input:
Question 1: {q1}
Question 2: {q2}

### Response:
"""

# Keep generation args here (not inside pipeline creation) to avoid config warnings
out = pipe(
    prompt,
    max_new_tokens=3,
    do_sample=False,
    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id,
    eos_token_id=tokenizer.eos_token_id
)

raw_answer = out[0]["generated_text"].strip()
first_token = re.sub(r"[^a-z]", "", raw_answer.split()[0].lower()) if raw_answer else ""

if first_token in ["yes", "duplicate"]:
    pred = "Duplicated"
elif first_token in ["no", "not"]:
    pred = "Not duplicated"
else:
    pred = f"Unclear output: {raw_answer}"

print("Final prediction:", pred)

[transformers] Both `max_new_tokens` (=3) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Final prediction: Duplicated
